# Project Aegis — Enhanced GNN Training (Colab GPU)

**What this does:**
1. Uploads your training data (JSON files)
2. Trains the improved GNN model on GPU (~5-15 min)
3. Downloads the trained model + real evaluation predictions

**Improvements over v1:**
- 25K+ training samples (was 2K)
- Focal Loss (was BCE)
- 4 GNN layers (was 3)
- Product + Difference interaction head (was concatenation)
- Label smoothing (0.05)
- Saves real predictions for chart generation

## Step 1: Setup & Install

In [ ]:
!pip install rdkit-pypi -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 2: Upload Training Data

Upload these 2 files from `web/data/gnn_training_enhanced/`:
- `train.json`
- `val.json`

In [ ]:
from google.colab import files
print("Upload train.json and val.json from web/data/gnn_training_enhanced/")
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

## Step 3: Model & Training Code

All model code is self-contained in this cell — no external imports needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import json
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple
from sklearn.metrics import (
    precision_recall_curve, auc, roc_auc_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from tqdm.notebook import tqdm

# ============================================================
# FEATURIZER
# ============================================================
ATOM_SYMBOLS = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'Si', 'B', 'Se']
DEGREES = [0, 1, 2, 3, 4, 5]
FORMAL_CHARGES = [-2, -1, 0, 1, 2]
NUM_HS = [0, 1, 2, 3, 4]
HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC,
]
ATOM_FEATURE_DIM = 40
EDGE_FEATURE_DIM = 8
MAX_ATOMS = 128

def _one_hot(value, choices):
    encoding = [0] * (len(choices) + 1)
    if value in choices:
        encoding[choices.index(value)] = 1
    else:
        encoding[-1] = 1
    return encoding

def get_atom_features(atom):
    features = []
    features.extend(_one_hot(atom.GetSymbol(), ATOM_SYMBOLS))
    features.extend(_one_hot(atom.GetDegree(), DEGREES))
    features.extend(_one_hot(atom.GetFormalCharge(), FORMAL_CHARGES))
    features.extend(_one_hot(atom.GetNumExplicitHs(), NUM_HS))
    features.extend(_one_hot(atom.GetHybridization(), HYBRIDIZATIONS))
    features.append(1.0 if atom.GetIsAromatic() else 0.0)
    features.append(1.0 if atom.IsInRing() else 0.0)
    return features

def get_bond_features(bond):
    features = []
    features.extend(_one_hot(bond.GetBondType(), BOND_TYPES))
    features.append(1.0 if bond.GetIsConjugated() else 0.0)
    features.append(1.0 if bond.IsInRing() else 0.0)
    features.append(1.0 if bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE else 0.0)
    return features

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    mol = Chem.RemoveHs(mol)
    num_atoms = mol.GetNumAtoms()
    if num_atoms == 0:
        return None
    effective_atoms = min(num_atoms, MAX_ATOMS)
    node_features = torch.zeros(MAX_ATOMS, ATOM_FEATURE_DIM)
    adjacency = torch.zeros(MAX_ATOMS, MAX_ATOMS)
    edge_features = torch.zeros(MAX_ATOMS, MAX_ATOMS, EDGE_FEATURE_DIM)
    node_mask = torch.zeros(MAX_ATOMS)
    for i in range(effective_atoms):
        atom = mol.GetAtomWithIdx(i)
        node_features[i] = torch.tensor(get_atom_features(atom), dtype=torch.float)
        node_mask[i] = 1.0
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if i >= MAX_ATOMS or j >= MAX_ATOMS:
            continue
        adjacency[i, j] = adjacency[j, i] = 1.0
        bf = torch.tensor(get_bond_features(bond), dtype=torch.float)
        edge_features[i, j] = edge_features[j, i] = bf
    for i in range(effective_atoms):
        adjacency[i, i] = 1.0
    return {'node_features': node_features, 'adjacency': adjacency,
            'edge_features': edge_features, 'node_mask': node_mask}

# ============================================================
# DATASET
# ============================================================
class DDIGraphDataset(Dataset):
    def __init__(self, data, use_binary=True):
        self.samples = []
        failed = 0
        for sample in tqdm(data, desc="Featurizing"):
            g1 = smiles_to_graph(sample['drug1_smiles'])
            g2 = smiles_to_graph(sample['drug2_smiles'])
            if g1 is None or g2 is None:
                failed += 1
                continue
            label = torch.tensor(sample.get('has_interaction', 0), dtype=torch.float)
            self.samples.append({
                'drug1_node_features': g1['node_features'],
                'drug1_adjacency': g1['adjacency'],
                'drug1_edge_features': g1['edge_features'],
                'drug1_node_mask': g1['node_mask'],
                'drug2_node_features': g2['node_features'],
                'drug2_adjacency': g2['adjacency'],
                'drug2_edge_features': g2['edge_features'],
                'drug2_node_mask': g2['node_mask'],
                'relation_label': label,
            })
        print(f"Featurized: {len(self.samples)}/{len(data)} (failed: {failed})")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

    @staticmethod
    def collate_fn(batch):
        keys = batch[0].keys()
        return {k: torch.stack([s[k] for s in batch]) for k in keys}

print("Featurizer & Dataset ready.")

In [ ]:
# ============================================================
# MODEL ARCHITECTURE (Enhanced v2)
# ============================================================

class EdgeConditionedGINConv(nn.Module):
    def __init__(self, in_dim, out_dim, edge_dim, eps=0.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU()
        )
        self.edge_transform = nn.Linear(edge_dim, in_dim)
        self.eps = nn.Parameter(torch.tensor([eps]))

    def forward(self, x, adj, edge_features, node_mask):
        B, N, D = x.shape
        eye = torch.eye(N, device=adj.device).unsqueeze(0)
        adj_no_self = adj * (1 - eye)
        edge_gates = torch.sigmoid(self.edge_transform(edge_features))
        x_expanded = x.unsqueeze(1).expand(B, N, N, D)
        gated_messages = x_expanded * edge_gates
        neighbor_sum = (gated_messages * adj_no_self.unsqueeze(-1)).sum(dim=2)
        out = (1 + self.eps) * x + neighbor_sum
        mask_flat = node_mask.view(-1).bool()
        out_flat = out.view(-1, D)
        if mask_flat.any():
            out_masked = self.mlp(out_flat[mask_flat])
            result = torch.zeros(B * N, out_masked.size(-1), device=x.device)
            result[mask_flat] = out_masked
        else:
            result = self.mlp(out_flat)
        return (result.view(B, N, -1)) * node_mask.unsqueeze(-1)


class MolecularGNNEncoder(nn.Module):
    def __init__(self, atom_dim=40, edge_dim=8, hidden_dim=256, num_layers=4, dropout=0.15, use_jk=True):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.use_jk = use_jk
        self.atom_embedding = nn.Linear(atom_dim, hidden_dim)
        self.atom_bn = nn.BatchNorm1d(hidden_dim)
        self.conv_layers = nn.ModuleList([
            EdgeConditionedGINConv(hidden_dim, hidden_dim, edge_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        if use_jk:
            self.jk_linear = nn.Linear(hidden_dim * (num_layers + 1), hidden_dim)
        self.output_dim = hidden_dim
        self.readout_dim = hidden_dim * 2  # mean + max

    def forward(self, node_features, adjacency, edge_features, node_mask):
        B, N, _ = node_features.shape
        mask_flat = node_mask.view(-1).bool()
        h_flat = node_features.view(-1, node_features.size(-1))
        if mask_flat.any():
            h_out = torch.zeros(B * N, self.hidden_dim, device=node_features.device)
            h_out[mask_flat] = F.relu(self.atom_bn(self.atom_embedding(h_flat[mask_flat])))
        else:
            h_out = F.relu(self.atom_bn(self.atom_embedding(h_flat)))
        h = (h_out.view(B, N, self.hidden_dim)) * node_mask.unsqueeze(-1)
        layer_outputs = [h]
        for conv in self.conv_layers:
            h = conv(h, adjacency, edge_features, node_mask)
            h = self.dropout(h)
            layer_outputs.append(h)
        if self.use_jk:
            h_jk = torch.cat(layer_outputs, dim=-1)
            h_flat2 = h_jk.view(-1, h_jk.size(-1))
            if mask_flat.any():
                h_proj = torch.zeros(B * N, self.output_dim, device=h.device)
                h_proj[mask_flat] = self.jk_linear(h_flat2[mask_flat])
            else:
                h_proj = self.jk_linear(h_flat2)
            h = (h_proj.view(B, N, self.output_dim)) * node_mask.unsqueeze(-1)
        # Global readout: mean + max
        mask = node_mask.unsqueeze(-1)
        h_mean = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        h_for_max = h.clone()
        h_for_max[mask.squeeze(-1) == 0] = float('-inf')
        h_max, _ = h_for_max.max(dim=1)
        h_max = torch.where(torch.isinf(h_max), torch.zeros_like(h_max), h_max)
        return torch.cat([h_mean, h_max], dim=-1)


class DDIInteractionHead(nn.Module):
    """Enhanced head: product + diff + sum instead of just concatenation."""
    def __init__(self, input_dim, hidden_dim=256, num_classes=1, dropout=0.15):
        super().__init__()
        single_dim = input_dim // 2
        combined_dim = single_dim * 3  # product + diff + sum
        self.dense1 = nn.Linear(combined_dim, hidden_dim)
        self.activation = nn.GELU()
        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.dense2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(hidden_dim // 2)
        self.classifier = nn.Linear(hidden_dim // 2, num_classes)

    def forward(self, x):
        half = x.size(-1) // 2
        d1, d2 = x[:, :half], x[:, half:]
        combined = torch.cat([d1 * d2, torch.abs(d1 - d2), d1 + d2], dim=-1)
        x = self.ln1(self.dropout1(self.activation(self.dense1(combined))))
        x = self.ln2(self.dropout2(self.activation(self.dense2(x))))
        return self.classifier(x)


class DDIGraphModel(nn.Module):
    def __init__(self, hidden_dim=256, num_layers=4, dropout=0.15):
        super().__init__()
        self.encoder = MolecularGNNEncoder(
            atom_dim=ATOM_FEATURE_DIM, edge_dim=EDGE_FEATURE_DIM,
            hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout
        )
        self.interaction_head = DDIInteractionHead(
            input_dim=self.encoder.readout_dim * 2,
            hidden_dim=hidden_dim, num_classes=1, dropout=dropout
        )

    def forward(self, d1_nf, d1_adj, d1_ef, d1_mask, d2_nf, d2_adj, d2_ef, d2_mask):
        emb1 = self.encoder(d1_nf, d1_adj, d1_ef, d1_mask)
        emb2 = self.encoder(d2_nf, d2_adj, d2_ef, d2_mask)
        combined = torch.cat([emb1, emb2], dim=-1)
        return self.interaction_head(combined)


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()

# Quick test
model = DDIGraphModel(hidden_dim=256, num_layers=4, dropout=0.15)
params = sum(p.numel() for p in model.parameters())
print(f"Model: {params:,} parameters")
print("Architecture ready.")

## Step 4: Load & Featurize Data

In [ ]:
with open('train.json') as f:
    train_data = json.load(f)
with open('val.json') as f:
    val_data = json.load(f)

print(f"Raw samples: train={len(train_data)}, val={len(val_data)}")

train_dataset = DDIGraphDataset(train_data)
val_dataset = DDIGraphDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          collate_fn=DDIGraphDataset.collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False,
                        collate_fn=DDIGraphDataset.collate_fn, num_workers=2, pin_memory=True)

print(f"\nDataloaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches")

## Step 5: Train!

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Hyperparameters
HIDDEN_DIM = 256
NUM_LAYERS = 4
DROPOUT = 0.15
LR = 5e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 80
PATIENCE = 15
LABEL_SMOOTHING = 0.05

model = DDIGraphModel(hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=len(train_loader) * EPOCHS, eta_min=LR * 0.01)

best_metric = 0.0
patience_counter = 0
history = []

print(f"\nStarting training: {EPOCHS} epochs, {sum(p.numel() for p in model.parameters()):,} params")
print(f"Train: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
print("="*60)

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        d1_nf = batch['drug1_node_features'].to(device)
        d1_adj = batch['drug1_adjacency'].to(device)
        d1_ef = batch['drug1_edge_features'].to(device)
        d1_mask = batch['drug1_node_mask'].to(device)
        d2_nf = batch['drug2_node_features'].to(device)
        d2_adj = batch['drug2_adjacency'].to(device)
        d2_ef = batch['drug2_edge_features'].to(device)
        d2_mask = batch['drug2_node_mask'].to(device)
        labels = batch['relation_label'].to(device)

        optimizer.zero_grad()
        logits = model(d1_nf, d1_adj, d1_ef, d1_mask, d2_nf, d2_adj, d2_ef, d2_mask)

        # Label smoothing
        smoothed = labels * (1 - LABEL_SMOOTHING) + (1 - labels) * LABEL_SMOOTHING
        loss = loss_fn(logits.squeeze(-1), smoothed)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        n_batches += 1

    avg_train_loss = train_loss / n_batches

    # --- Evaluate ---
    model.eval()
    all_labels, all_scores, all_logits_list = [], [], []
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for batch in val_loader:
            d1_nf = batch['drug1_node_features'].to(device)
            d1_adj = batch['drug1_adjacency'].to(device)
            d1_ef = batch['drug1_edge_features'].to(device)
            d1_mask = batch['drug1_node_mask'].to(device)
            d2_nf = batch['drug2_node_features'].to(device)
            d2_adj = batch['drug2_adjacency'].to(device)
            d2_ef = batch['drug2_edge_features'].to(device)
            d2_mask = batch['drug2_node_mask'].to(device)
            labels = batch['relation_label'].to(device)

            logits = model(d1_nf, d1_adj, d1_ef, d1_mask, d2_nf, d2_adj, d2_ef, d2_mask)
            loss = loss_fn(logits.squeeze(-1), labels)
            val_loss += loss.item()
            n_val += 1

            scores = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
            all_labels.extend(labels.cpu().numpy().tolist())
            all_scores.extend(scores.tolist())
            all_logits_list.extend(logits.squeeze(-1).cpu().numpy().tolist())

    y_true = np.array(all_labels)
    y_scores = np.array(all_scores)
    y_pred = (y_scores >= 0.5).astype(int)

    # Metrics
    try:
        prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_scores)
        pr_auc = auc(rec_curve, prec_curve)
    except:
        pr_auc = 0.0
    try:
        roc = roc_auc_score(y_true, y_scores)
    except:
        roc = 0.0
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = np.mean(y_true == y_pred)

    epoch_data = {
        'epoch': epoch + 1, 'train_loss': avg_train_loss,
        'val_loss': val_loss / max(n_val, 1),
        'pr_auc': pr_auc, 'roc_auc': roc,
        'precision': prec, 'recall': rec, 'f1': f1, 'accuracy': acc
    }
    history.append(epoch_data)

    # Print progress
    print(f"Epoch {epoch+1:3d} | Loss: {avg_train_loss:.4f} | Val: {val_loss/max(n_val,1):.4f} | "
          f"PR-AUC: {pr_auc:.4f} | ROC: {roc:.4f} | P: {prec:.3f} R: {rec:.3f} F1: {f1:.3f} Acc: {acc:.3f}")

    # Best model
    if pr_auc > best_metric:
        best_metric = pr_auc
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {'hidden_dim': HIDDEN_DIM, 'num_gnn_layers': NUM_LAYERS,
                       'dropout_rate': DROPOUT, 'use_jumping_knowledge': True,
                       'max_atoms': MAX_ATOMS, 'use_binary': True, 'num_relation_classes': 1},
            'metrics': epoch_data,
        }, 'gnn_best_model.pt')
        print(f"  >>> New best PR-AUC: {pr_auc:.4f} (saved)")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE. Best PR-AUC: {best_metric:.4f}")
print(f"{'='*60}")

## Step 6: Final Evaluation & Save Predictions

In [ ]:
# Load best model and run final evaluation
checkpoint = torch.load('gnn_best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_labels, all_scores = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final eval"):
        d1_nf = batch['drug1_node_features'].to(device)
        d1_adj = batch['drug1_adjacency'].to(device)
        d1_ef = batch['drug1_edge_features'].to(device)
        d1_mask = batch['drug1_node_mask'].to(device)
        d2_nf = batch['drug2_node_features'].to(device)
        d2_adj = batch['drug2_adjacency'].to(device)
        d2_ef = batch['drug2_edge_features'].to(device)
        d2_mask = batch['drug2_node_mask'].to(device)
        labels = batch['relation_label']
        logits = model(d1_nf, d1_adj, d1_ef, d1_mask, d2_nf, d2_adj, d2_ef, d2_mask)
        scores = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
        all_labels.extend(labels.numpy().tolist())
        all_scores.extend(scores.tolist())

y_true = np.array(all_labels)
y_scores = np.array(all_scores)
y_pred = (y_scores >= 0.5).astype(int)

# Compute final metrics
prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(rec_curve, prec_curve)
roc = roc_auc_score(y_true, y_scores)
cm = confusion_matrix(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
acc = np.mean(y_true == y_pred)

print(f"\n{'='*50}")
print(f"FINAL EVALUATION (Best Model)")
print(f"{'='*50}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"ROC-AUC:   {roc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Accuracy:  {acc:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0][0]:5d}  FP={cm[0][1]:5d}")
print(f"  FN={cm[1][0]:5d}  TP={cm[1][1]:5d}")

# Save predictions for real chart generation
eval_data = {
    'y_true': all_labels,
    'y_scores': all_scores,
    'n_samples': len(all_labels),
    'metrics': {
        'pr_auc': float(pr_auc), 'roc_auc': float(roc),
        'precision': float(prec), 'recall': float(rec),
        'f1': float(f1), 'accuracy': float(acc),
        'confusion_matrix': cm.tolist(),
    },
    'roc_curve': {
        'fpr': None, 'tpr': None  # filled below
    },
    'pr_curve': {
        'precision': prec_curve.tolist(),
        'recall': rec_curve.tolist(),
    },
}

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_scores)
eval_data['roc_curve'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist()}

with open('evaluation_predictions.json', 'w') as f:
    json.dump(eval_data, f)
print(f"\nSaved evaluation_predictions.json ({len(all_labels)} samples)")

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f"Saved training_history.json ({len(history)} epochs)")

# Save training results
results = {
    'best_metric': float(best_metric),
    'config': checkpoint['config'],
    'train_samples': len(train_dataset),
    'val_samples': len(val_dataset),
    'device': str(device),
    'data_sources': 'neo4j+ddi_corpus+twosides',
    'version': 'enhanced_v2',
    'final_metrics': eval_data['metrics'],
}
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved training_results.json")

## Step 7: Download Results

Downloads 4 files — put them back in `web/models/gnn/`:
- `gnn_best_model.pt` — trained model weights
- `evaluation_predictions.json` — real y_true/y_scores for chart generation
- `training_history.json` — loss curves per epoch
- `training_results.json` — final metrics summary

In [ ]:
from google.colab import files

for fname in ['gnn_best_model.pt', 'evaluation_predictions.json',
              'training_history.json', 'training_results.json']:
    files.download(fname)
    print(f"Downloaded: {fname}")

print("\nDone! Place these files in: web/models/gnn/")
print("Then run the real visualization generator to create charts from actual predictions.")